# QRculus · 다변수 QR 판독 성공률 예측 + 변수 제거 실험

Google Colab에서 **런타임 → 모두 실행**을 누르세요. QRculus v2 실험 원자료를 다운로드하고 **가림률, 위치 탐지 패턴 가림률, 가림 형태, 오류 정정 등급**을 활용해 모델 4종을 평가합니다.

이어지는 **변수 제거 실험(Ablation Study)**에서는 같은 모델·훈련/검증 데이터·평가 기준으로 **1개 → 2개 → 3개 → 4개 입력 변수**를 차례로 추가해 예측 오차가 정말 감소하는지 확인합니다.

- 입력 데이터: `raw/trials.csv` (112,400회), `raw/mask_orders.csv` (2,000개 가림 순서), `payload_split.csv`.
- 원래 연구의 QR 원문 기준 분할: **학습 80개 / 검증 20개** (동일 QR의 반복 측정은 같은 집합).
- 위치 탐지 패턴: 모서리 3개 × 7×7칸 = 147칸. **가림 순서에서 실제 가려진 탐지 패턴 칸 수**를 계산합니다.
- 평가: **116개 조건의 성공률 MAE(%p)**, 개별 판독의 Brier 점수, Log Loss. 작은 수치가 좋습니다.
- 주의: 기존 가림률 단계에서의 새 QR 원문 예측 성능입니다. 새로운 가림률 간격에 대한 일반화는 별도 실험이 필요합니다.


In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib requests

## 1. 원본 데이터 다운로드 · 탐지 패턴 가림률 계산 · 학습 및 검증

In [ ]:
"""QRculus v2 trial-level multivariate model comparison (Colab).
Uses original train/validation payload split and reconstructs finder coverage
from saved mask orders (same method as src/qr_calculus/pink_analysis.py).
"""
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import log_loss, brier_score_loss
import matplotlib.pyplot as plt

ROOT = 'https://raw.githubusercontent.com/hwanghj09/QRculus/main/results/pink_v2/'
OUT = Path('qrculus_multivariate_results')
OUT.mkdir(parents=True, exist_ok=True)

# Download all three source files: original trials, saved mask order, official split.
files_to_read = {
    'trials.csv': ROOT + 'raw/trials.csv',
    'mask_orders.csv': ROOT + 'raw/mask_orders.csv',
    'payload_split.csv': ROOT + 'payload_split.csv',
}
for filename, url in files_to_read.items():
    path = Path(filename)
    if not path.exists():
        print(f'Downloading {filename} ...')
        pd.read_csv(url, low_memory=False).to_csv(path, index=False)

df = pd.read_csv('trials.csv', low_memory=False)
orders = pd.read_csv('mask_orders.csv', low_memory=False)
splits = pd.read_csv('payload_split.csv')
print('Trial columns:', ', '.join(df.columns))
print(f'{len(df):,} trials, {len(orders):,} mask orders, {len(splits):,} payloads')

required_trials = {'payload_id','repeat_id','mask_type','level','target_pct',
                   'actual_pct','covered_modules','covered_reserved','success'}
required_orders = {'payload_id','repeat_id','mask_type','cells_row_major_index'}
if missing := required_trials.difference(df.columns):
    raise ValueError(f'Missing trial fields: {sorted(missing)}')
if missing := required_orders.difference(orders.columns):
    raise ValueError(f'Missing mask-order fields: {sorted(missing)}')
if not {'payload_id','split'}.issubset(splits.columns):
    raise ValueError('payload_split.csv must contain payload_id and split')

# These exact 7x7 squares are used in the existing QRculus analysis.
SIDE = 21
finder_mask = np.zeros((SIDE, SIDE), dtype=bool)
finder_mask[:7, :7] = True                 # upper left
finder_mask[:7, -7:] = True                # upper right
finder_mask[-7:, :7] = True                # lower left
finder_flat = finder_mask.ravel()
assert finder_flat.sum() == 147

target_values = sorted(int(v) for v in df.loc[df['target_pct'] > 0, 'target_pct'].unique())
ks = {t: round(SIDE * SIDE * t / 100) for t in target_values}
lookup_rows = []
for row in orders.itertuples(index=False):
    order = np.fromstring(row.cells_row_major_index, sep=' ', dtype=np.int16)
    if order.size != SIDE * SIDE or np.unique(order).size != SIDE * SIDE or order.min() < 0 or order.max() >= SIDE * SIDE:
        raise ValueError(f'Invalid mask order for {row.payload_id}/{row.repeat_id}/{row.mask_type}')
    cumulative = np.cumsum(finder_flat[order])
    for target, k in ks.items():
        lookup_rows.append((row.payload_id, int(row.repeat_id), row.mask_type,
                            target, int(cumulative[k-1])))
lookup = pd.DataFrame(lookup_rows, columns=['payload_id','repeat_id','mask_type',
                                             'target_pct','covered_finder'])

df = df.merge(lookup, on=['payload_id','repeat_id','mask_type','target_pct'],
              how='left', validate='many_to_one')
missing_finder = (df['target_pct'] > 0) & df['covered_finder'].isna()
if missing_finder.any():
    raise ValueError(f'{missing_finder.sum():,} nonzero trials do not match saved mask orders')
df['covered_finder'] = df['covered_finder'].fillna(0).astype('int16')
if not (df['covered_finder'] <= df['covered_reserved']).all():
    raise ValueError('Finder coverage is inconsistent with reserved-cell counts')
df['finder_coverage_pct'] = df['covered_finder'] * (100 / 147)

# Use the ORIGINAL split from the experiment (not a new random split).
df = df.merge(splits[['payload_id','split']], on='payload_id', how='left', validate='many_to_one')
if df['split'].isna().any():
    raise ValueError('Some payloads are absent from payload_split.csv')
training = df['split'].eq('train')
validation = df['split'].eq('validation')
if not (training | validation).all() or not training.any() or not validation.any():
    raise ValueError('Expected train/validation labels in payload_split.csv')
print(f'Train {training.sum():,} trials / {df.loc[training,"payload_id"].nunique()} payloads; '
      f'validation {validation.sum():,} trials / {df.loc[validation,"payload_id"].nunique()} payloads')

# Sanity checks: 0% occlusion -> 0% finder occlusion, maximum 100% finder.
assert df.loc[df['covered_modules']==0, 'covered_finder'].eq(0).all()
assert df['finder_coverage_pct'].between(0, 100).all()
X = df[['actual_pct', 'finder_coverage_pct', 'mask_type', 'level']].copy()
X.columns = ['occlusion_pct', 'finder_coverage_pct', 'mask_type', 'ecc_level']
y = pd.to_numeric(df['success'], errors='raise').astype('int8')
assert y.isin([0, 1]).all()

numeric = ['occlusion_pct', 'finder_coverage_pct']
categorical = ['mask_type','ecc_level']
preprocessor = ColumnTransformer([
    ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
])
models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, C=1.0),
    'RandomForest': RandomForestClassifier(n_estimators=120, min_samples_leaf=12,
                                           n_jobs=-1, random_state=2026),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=120, min_samples_leaf=12,
                                       n_jobs=-1, random_state=2026),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=160,
                                                           l2_regularization=3,
                                                           random_state=2026),
}

results, predictions = [], []
for name, estimator in models.items():
    print(f'Training {name} ...')
    pipe = make_pipeline(preprocessor, estimator)
    pipe.fit(X.loc[training], y.loc[training])
    probs = pipe.predict_proba(X.loc[validation])[:, 1]
    results.append({'model': name,
                    'trial_brier': brier_score_loss(y.loc[validation], probs),
                    'trial_log_loss': log_loss(y.loc[validation], probs)})
    z = X.loc[validation].copy()
    z['actual'] = y.loc[validation].to_numpy()
    z['predicted'] = probs
    z['model'] = name
    z['payload_id'] = df.loc[validation, 'payload_id'].to_numpy()
    predictions.append(z)

metrics = pd.DataFrame(results).sort_values('trial_brier')
metrics.to_csv(OUT/'metrics.csv', index=False)
pred = pd.concat(predictions, ignore_index=True)
pred.to_csv(OUT/'predictions.csv', index=False)
print('\nTrial-level evaluation (lower is better):\n', metrics.to_string(index=False))

# Percentage-point errors of success probabilities aggregated by experimental condition.
# These groups use only coverage / shape / ECC so comparisons match prior study.
groupcols = ['occlusion_pct','mask_type','ecc_level']
grp = pred.groupby(['model'] + groupcols, as_index=False)[['actual','predicted']].mean()
grp['abs_error_pp'] = 100 * (grp['actual'] - grp['predicted']).abs()
score = grp.groupby('model', as_index=False)['abs_error_pp'].mean().sort_values('abs_error_pp')
score.to_csv(OUT/'group_mae.csv', index=False)
grp.to_csv(OUT/'group_predictions.csv', index=False)
print('\nGrouped success-rate MAE (%p):\n', score.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(score['model'], score['abs_error_pp'])
ax.set_ylabel('Group MAE (percentage points)')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(OUT/'model_comparison.png', dpi=180)
plt.show()
print('\nSaved results to:', OUT.resolve())


## 2. 기존 4변수 모델 결과
먼저 네 변수를 모두 사용하는 기존 모델들의 성능을 확인합니다.

In [ ]:
from IPython.display import display, Image
display(metrics)
display(score)
display(Image(filename=str(OUT/'model_comparison.png')))


## 3. 변수 제거 실험 (Ablation Study)

**같은 학습 80개 / 검증 20개 QR**, **같은 모델 설정**, **같은 116개 조건**, **같은 평가 지표**에서 입력 변수만 바꿉니다.

| 단계 | 입력 변수 |
|---|---|
| ① | 전체 가림률 |
| ② | 전체 가림률 + 오류 정정 등급 |
| ③ | 전체 가림률 + 오류 정정 등급 + 가림 형태 |
| ④ | 전체 가림률 + 오류 정정 등급 + 가림 형태 + 위치 탐지 패턴 가림률 |

각 단계마다 로지스틱 회귀, 랜덤 포레스트, Extra Trees, 히스토그램 기반 경사 부스팅을 비교합니다(총 16개 조합). ④의 기존 예측은 위 2단계 결과를 **재사용**하므로 중복 학습을 하지 않습니다.

**해석:** MAE가 감소하면 변수를 추가했을 때 해당 모델의 검증 예측이 개선된 것입니다. 오히려 MAE가 커질 수도 있으며, 변수의 인과적 영향이나 다른 가림률·버전에서도 성립한다는 뜻은 아닙니다.

In [ ]:
# === Ablation Study: 4 feature sets × 4 estimators ===
from sklearn.base import clone
from sklearn.metrics import brier_score_loss, log_loss

FEATURE_SETS = {
    '1_cover': ['occlusion_pct'],
    '2_cover_ecc': ['occlusion_pct', 'ecc_level'],
    '3_cover_ecc_shape': ['occlusion_pct', 'ecc_level', 'mask_type'],
    '4_all_features': ['occlusion_pct', 'ecc_level', 'mask_type', 'finder_coverage_pct'],
}
FEATURE_LABELS = {
    '1_cover': '① 가림률',
    '2_cover_ecc': '② + 오류 정정',
    '3_cover_ecc_shape': '③ + 가림 형태',
    '4_all_features': '④ + 탐지 패턴',
}
GROUP_KEYS = ['occlusion_pct', 'mask_type', 'ecc_level']
Y_VALID = y.loc[validation].to_numpy(dtype=np.int8)
BASE_VALID = X.loc[validation, GROUP_KEYS].reset_index(drop=True)
assert len(BASE_VALID) == len(Y_VALID) and len(Y_VALID) > 0

# Consistent evaluation: each experimental condition receives equal weight
# in group MAE (116 groups). Brier/log loss use every individual attempt.
def evaluate_ablation(probs, feature_set, name):
    probs = np.asarray(probs, dtype=float)
    if probs.shape != Y_VALID.shape or not np.isfinite(probs).all():
        raise ValueError(f'Invalid probabilities: {feature_set}/{name}')
    probs = np.clip(probs, 1e-12, 1-1e-12)
    group = BASE_VALID.copy()
    group['actual'] = Y_VALID
    group['predicted'] = probs
    group = group.groupby(GROUP_KEYS, as_index=False)[['actual','predicted']].mean()
    group['abs_error_pp'] = (group['actual']-group['predicted']).abs()*100
    group['feature_set'] = feature_set
    group['model'] = name
    low = group.loc[(group.occlusion_pct>0) & (group.occlusion_pct<=10), 'abs_error_pp']
    result = {
        'feature_set': feature_set,
        'feature_label': FEATURE_LABELS[feature_set],
        'n_features': len(FEATURE_SETS[feature_set]),
        'model': name,
        'group_mae_pp': group['abs_error_pp'].mean(),
        'low_coverage_mae_pp': low.mean(),
        'trial_brier': brier_score_loss(Y_VALID, probs),
        'trial_log_loss': log_loss(Y_VALID, probs, labels=[0,1]),
        'n_groups': len(group),
        'n_trials': len(Y_VALID),
    }
    return result, group

ablation_records, ablation_group_frames = [], []
for feature_set, feature_cols in FEATURE_SETS.items():
    print(f'\n{FEATURE_LABELS[feature_set]}: {", ".join(feature_cols)}')
    for name, estimator in models.items():
        if feature_set == '4_all_features':
            # The previous section fitted exactly these four features with
            # the same payload split and estimator settings. Reuse probabilities.
            existing = pred.loc[pred['model'].eq(name)]
            if len(existing) != len(Y_VALID) or not np.array_equal(existing['actual'].to_numpy(), Y_VALID):
                raise ValueError(f'Existing predictions differ from validation order: {name}')
            probs = existing['predicted'].to_numpy()
            print(f'  {name}: reused existing full-model predictions')
        else:
            numeric_cols = [c for c in feature_cols if c in ('occlusion_pct','finder_coverage_pct')]
            category_cols = [c for c in feature_cols if c in ('ecc_level','mask_type')]
            transformers = []
            if numeric_cols:
                transformers.append(('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric_cols))
            if category_cols:
                transformers.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), category_cols))
            feature_preprocessor = ColumnTransformer(transformers)
            fitted = make_pipeline(feature_preprocessor, clone(estimator))
            print(f'  Fitting {name} ...')
            fitted.fit(X.loc[training, feature_cols], y.loc[training])
            probs = fitted.predict_proba(X.loc[validation, feature_cols])[:, 1]
        record, group_frame = evaluate_ablation(probs, feature_set, name)
        if record['n_groups'] != 116:
            raise ValueError(f'Expected 116 groups, found {record["n_groups"]}')
        ablation_records.append(record)
        ablation_group_frames.append(group_frame)
        print(f'    MAE {record["group_mae_pp"]:.4f}%p | Brier {record["trial_brier"]:.5f}')

ablation = pd.DataFrame(ablation_records)
ablation_groups = pd.concat(ablation_group_frames, ignore_index=True)
ablation.to_csv(OUT/'ablation_metrics.csv', index=False)
ablation_groups.to_csv(OUT/'ablation_group_predictions.csv', index=False)

# Stage ④ must reproduce exactly the original four-variable benchmark.
existing_score = dict(zip(score['model'], score['abs_error_pp']))
for _, record in ablation.loc[ablation.feature_set.eq('4_all_features')].iterrows():
    if not np.isclose(record.group_mae_pp, existing_score[record.model], rtol=0, atol=1e-8):
        raise AssertionError('Full-feature results disagree with original benchmark')

wide = ablation.pivot(index='feature_set', columns='model', values='group_mae_pp')
wide = wide.reindex(list(FEATURE_SETS))
wide.to_csv(OUT/'ablation_group_mae_table.csv')

rows = []
for name in models:
    records = ablation.loc[ablation.model.eq(name)].set_index('feature_set').loc[list(FEATURE_SETS)]
    values = records['group_mae_pp'].to_numpy()
    for i, (stage, mae) in enumerate(zip(FEATURE_SETS, values)):
        rows.append({
            'model':name, 'feature_set':stage, 'group_mae_pp':mae,
            'delta_from_previous_pp':(values[i-1]-mae) if i else np.nan,
            'improvement_vs_cover_only_pp':values[0]-mae,
        })
deltas = pd.DataFrame(rows)
deltas.to_csv(OUT/'ablation_improvement.csv', index=False)

print('\n=== 변수 조합별 성공률 MAE(%p) — 작을수록 정확 ===')
display(wide.round(4))
print('\n=== 각 모델에서 변수 추가 시 개선(+)/악화(-) (%p) ===')
display(deltas.round(4))

fig, ax = plt.subplots(figsize=(10, 5.5))
for name in models:
    vals = wide[name].to_numpy()
    ax.plot(np.arange(4), vals, '-o', label=name)
ax.set_xticks(np.arange(4), [FEATURE_LABELS[k] for k in FEATURE_SETS])
ax.set_ylabel('Group success-rate MAE (percentage points)')
ax.set_xlabel('Variables included in the model')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(OUT/'ablation_mae_line.png', dpi=170)
plt.show()

fig, ax = plt.subplots(figsize=(10, 5.5))
for name in models:
    subset = ablation.loc[ablation.model.eq(name)].set_index('feature_set').loc[list(FEATURE_SETS)]
    ax.plot(np.arange(4), subset['trial_brier'].to_numpy(), '-o', label=name)
ax.set_xticks(np.arange(4), [FEATURE_LABELS[k] for k in FEATURE_SETS])
ax.set_ylabel('Brier score (individual trial; lower is better)')
ax.set_xlabel('Variables included in the model')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(OUT/'ablation_brier_line.png', dpi=170)
plt.show()

# Inspect which QR conditions benefit most from finder-coverage information.
best_full = ablation.loc[ablation.feature_set.eq('4_all_features')].sort_values('group_mae_pp').iloc[0]
best_model = best_full['model']
before = ablation_groups.loc[(ablation_groups.model.eq(best_model)) &
                             (ablation_groups.feature_set.eq('3_cover_ecc_shape')),
                             GROUP_KEYS+['abs_error_pp']].rename(columns={'abs_error_pp':'before_mae_pp'})
after = ablation_groups.loc[(ablation_groups.model.eq(best_model)) &
                            (ablation_groups.feature_set.eq('4_all_features')),
                            GROUP_KEYS+['abs_error_pp']].rename(columns={'abs_error_pp':'after_mae_pp'})
impact = before.merge(after, on=GROUP_KEYS, validate='one_to_one')
impact['finder_addition_improvement_pp'] = impact['before_mae_pp']-impact['after_mae_pp']
impact.sort_values('finder_addition_improvement_pp', ascending=False).to_csv(OUT/'finder_feature_condition_impact.csv', index=False)
print(f'\n④ 전체 변수 기준 1위 모델: {best_model} | MAE {best_full.group_mae_pp:.4f}%p')
print('탐지 패턴 변수 추가 후 오차가 가장 크게 줄어든 조건 10개:')
display(impact.sort_values('finder_addition_improvement_pp',ascending=False).head(10).round(4))

best_stage_each = ablation.sort_values('group_mae_pp').groupby('feature_set',as_index=False).first()
best_stage_each = best_stage_each.set_index('feature_set').loc[list(FEATURE_SETS)].reset_index()
best_stage_each.to_csv(OUT/'ablation_stage_winners.csv', index=False)
print('\n각 변수 조합에서 가장 정확한 모델:')
display(best_stage_each[['feature_label','model','group_mae_pp','trial_brier']].round(5))

# Report generated from measurements, not an assumed improvement.
base = best_stage_each.iloc[0]
final = best_stage_each.iloc[-1]
change = base.group_mae_pp - final.group_mae_pp
report = f'''# QRculus 변수 제거 실험 결과

- 학습: 기존 train 원문 80개, 검증: 기존 validation 원문 20개 (검증 22,480건).
- 같은 116개 실험 조건의 성공률 MAE(%p)를 동일한 가중치로 평균함.
- 가림률만 입력한 단계의 최고 모델: **{base.model}**, MAE **{base.group_mae_pp:.4f}%p**.
- 4변수 단계의 최고 모델: **{final.model}**, MAE **{final.group_mae_pp:.4f}%p**.
- 차이(양수=4변수에서 개선): **{change:+.4f}%p**.
- 다른 지표: 개별 판독의 Brier 점수와 log loss도 함께 비교.
- 이 결과는 같은 15개 목표 가림률(0 포함)과 QR 버전 1, OpenCV 판독기에서 원문만 바뀔 때의 성능임.
- 여러 모델을 같은 검증 데이터로 비교했으므로, 최종 일반화 성능을 확정하려면 추가 독립 검증이 필요함.

## 변수 조합별 우수 모델

| 입력 변수 단계 | 모델 | 그룹 MAE(%p) | Brier |
|---|---|---:|---:|
'''
for _, row in best_stage_each.iterrows():
    report += f'| {row.feature_label} | {row.model} | {row.group_mae_pp:.4f} | {row.trial_brier:.5f} |\n'
(OUT/'ablation_summary.md').write_text(report,encoding='utf-8')
print('\n생성 파일:', ', '.join(p.name for p in sorted(OUT.glob('ablation*'))))


## 4. 결과 파일 다운로드

모델별 4단계 변수 조합의 결과표(`ablation_metrics.csv`), 성능 변화(`ablation_improvement.csv`), 조건별 영향(`finder_feature_condition_impact.csv`), 그래프 2장 및 자동 작성된 요약 보고서(`ablation_summary.md`)를 ZIP에 담습니다. 기존 전체 변수 모델 비교 결과도 함께 보존합니다.

In [ ]:
from IPython.display import display
import shutil
print('분석 결과:', OUT.resolve())
print('변수 제거 실험 순위:')
display(best_stage_each[['feature_label','model','group_mae_pp','trial_brier']].round(5))
archive = shutil.make_archive('qrculus_multivariate_ablation_results', 'zip', root_dir='.', base_dir=str(OUT))
print('ZIP:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Colab 밖에서는 생성된 ZIP 파일을 직접 다운로드하세요.')
